In [34]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
#from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits, load_wine, load_breast_cancer


In [35]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("first_mlflow_exp")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1788774234620, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788774234620, lifecycle_stage='active', name='first_mlflow_exp', tags={}, trace_location=None, workspace='default'>

In [36]:
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [37]:
n_estimators_list = [50, 100, 200]
max_depth_list = [3, 5, 10]

for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        with mlflow.start_run():
            model = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=42
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)

            mlflow.log_param("n_estimators", n_estimators)
            mlflow.log_param("max_depth", max_depth)
            mlflow.log_metric("accuracy", accuracy)
            mlflow.sklearn.log_model(model, name="model")

            print(f"Run complete: n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy}")


Run complete: n_estimators=50, max_depth=3, accuracy=0.8833333333333333
🏃 View run glamorous-kit-150 at: http://127.0.0.1:5000/#/experiments/1/runs/4c9091a604804e2bb2f6d15130c86556
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run complete: n_estimators=50, max_depth=5, accuracy=0.9444444444444444
🏃 View run bittersweet-toad-736 at: http://127.0.0.1:5000/#/experiments/1/runs/77dfa63409cd4a368ae666ce6b67ee7f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run complete: n_estimators=50, max_depth=10, accuracy=0.9666666666666667
🏃 View run skittish-fish-936 at: http://127.0.0.1:5000/#/experiments/1/runs/c025aa2255754e62a8294b94de89b68a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run complete: n_estimators=100, max_depth=3, accuracy=0.8972222222222223
🏃 View run glamorous-kite-366 at: http://127.0.0.1:5000/#/experiments/1/runs/6686ba98910e49cb8e9629ab0de12095
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run complete: n_estimators=100

In [38]:
client = MlflowClient()
experiment = client.get_experiment_by_name("first_mlflow_exp")

best_run = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=1
)[0]

print("Best run ID:", best_run.info.run_id)
print("Best accuracy:", best_run.data.metrics["accuracy"])
print("Best params:", best_run.data.params)


Best run ID: d44e75a89f0047ccad63fdc5c253fd49
Best accuracy: 1.0
Best params: {'n_estimators': '200', 'max_depth': '10'}


In [ ]:
# -----------------------------
# Reload best model with a name
# -----------------------------
model_name = "DigitModel"

best_model_uri = f"runs:/{best_run.info.run_id}/DigitModel"
best_model = mlflow.sklearn.load_model(best_model_uri)

# Ensure dataset matches
predictions = best_model.predict(X_test)   # X_test from load_digits
print(f"Model name: {model_name}")
print("Best run ID:", best_run.info.run_id)
print("Predictions from best model:", predictions[:10])


ValueError: X has 64 features, but RandomForestClassifier is expecting 4 features as input.

In [ ]:
# -----------------------------
# Load the best model
# -----------------------------
best_model = mlflow.sklearn.load_model(f"runs:/{best_run.info.run_id}/model")
print("✅ Best model loaded:", best_model)

# -----------------------------
# Register the best model
# -----------------------------
print("📦 Registering the best model...")

result = mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/model",
    name=model_name
)

print("✅ Model registered!")
print("Model name   :", result.name)
print("Model version:", result.version)

✅ Best model loaded: RandomForestClassifier(max_depth=10, n_estimators=200, random_state=42)
📦 Registering the best model...


Registered model 'IrisBestModel' already exists. Creating a new version of this model...
2026/09/09 07:13:09 WARNING mlflow.tracking._model_registry.fluent: Run with id d44e75a89f0047ccad63fdc5c253fd49 has no artifacts at artifact path 'model', registering model based on models:/m-0e9f94e33f9b470182e9a2354e08a094 instead
2026/09/09 07:13:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IrisBestModel, version 2
Created version '2' of model 'IrisBestModel'.


✅ Model registered!
Model name   : IrisBestModel
Model version: 2


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -----------------------------
# Validate the best model
# -----------------------------
y_pred = best_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Validation Accuracy:", accuracy)

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Validation Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


In [ ]:
# Promote to Production
client.transition_model_version_stage(
    name=model_name,
    version=result.version,
    stage="Production",
    archive_existing_versions=True
)

print("✅ Model promoted to Production!")

# Optional: Serve the model via REST API
# Run this in terminal:
# mlflow models serve -m "models:/my_model/Production" -p 5000 --no-conda

✅ Model promoted to Production!


C:\Users\Abhidnya\AppData\Local\Temp\ipykernel_24228\2286980690.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [ ]:
# -----------------------------
# Import required libraries
# -----------------------------
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.datasets import load_digits

# -----------------------------
# Configure MLflow
# -----------------------------
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("digits_exp")   # ✅ give dataset-specific experiment name

# -----------------------------
# Load and prepare the dataset
# -----------------------------
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# Define hyperparameter values to try
# -----------------------------
n_estimators_list = [50, 100, 200]
max_depth_list = [3, 5, 10]

# -----------------------------
# Run multiple experiments
# -----------------------------
for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        with mlflow.start_run():
            model = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=42
            )
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)

            # Log parameters and metrics
            mlflow.log_param("n_estimators", n_estimators)
            mlflow.log_param("max_depth", max_depth)
            mlflow.log_metric("accuracy", accuracy)

            # Log model
            mlflow.sklearn.log_model(model, name="DigitModel")

            print(f"Run complete: n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy}")

# -----------------------------
# Select best run
# -----------------------------
client = MlflowClient()
experiment = client.get_experiment_by_name("digits_exp")

best_run = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=1
)[0]

print("Best run ID:", best_run.info.run_id)
print("Best accuracy:", best_run.data.metrics["accuracy"])
print("Best params:", best_run.data.params)

# -----------------------------
# Reload best model with a name
# -----------------------------
model_name = "DigitModel"
best_model_uri = f"runs:/{best_run.info.run_id}/DigitModel"
best_model = mlflow.sklearn.load_model(best_model_uri)

predictions = best_model.predict(X_test)
print(f"Model name: {model_name}")
print("Best run ID:", best_run.info.run_id)
print("Predictions from best model:", predictions[:10])

# -----------------------------
# Register the best model
# -----------------------------
print("📦 Registering the best model...")
result = mlflow.register_model(
    model_uri=best_model_uri,
    name=model_name
)

print("✅ Model registered!")
print("Model name   :", result.name)
print("Model version:", result.version)

# -----------------------------
# Validate the best model
# -----------------------------
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Validation Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# -----------------------------
# Promote to Production
# -----------------------------
client.transition_model_version_stage(
    name=model_name,
    version=result.version,
    stage="Production",
    archive_existing_versions=True
)

print("✅ Model promoted to Production!")


Run complete: n_estimators=50, max_depth=3, accuracy=0.8833333333333333
🏃 View run bemused-horse-77 at: http://127.0.0.1:5000/#/experiments/3/runs/fec4d4af36434f239062ba608449d946
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Run complete: n_estimators=50, max_depth=5, accuracy=0.9444444444444444
🏃 View run righteous-foal-506 at: http://127.0.0.1:5000/#/experiments/3/runs/ba855cd1abe34f20b12b16036007063e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Run complete: n_estimators=50, max_depth=10, accuracy=0.9666666666666667
🏃 View run powerful-flea-374 at: http://127.0.0.1:5000/#/experiments/3/runs/f7d14213076e4dfaa98ef29567f1f930
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Run complete: n_estimators=100, max_depth=3, accuracy=0.8972222222222223
🏃 View run likeable-ram-968 at: http://127.0.0.1:5000/#/experiments/3/runs/b8ff381433dc49c9a720f1d138a2e359
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Run complete: n_estimators=100, max

MlflowException: Failed to download artifacts from path 'model', please ensure that the path is correct.